# 05 — LLM as Judge

## Why this notebook exists

In **notebook 04** we built a regression gate using only deterministic graders — exact match, `contains`, structured validation, golden outputs. Those graders are fast, free, and fully reproducible. But they break down the moment the task produces *open-ended text*: two summaries can be equally faithful and concise while sharing almost no words. An `exact_match` grader marks one correct and one wrong based purely on whether the string matches a reference. A `contains` grader tells us a keyword appears, not whether the output is actually good.

LLM-as-judge plugs that gap: we ask a language model to score an output against an explicit rubric, returning a structured score and a rationale we can read. This notebook introduces the technique, shows you how to wire it into the harness from notebook 03 as just another grader, and — critically — shows you where it goes wrong and how to check whether your judge can be trusted.

This is the **first notebook in the series that requires an OpenAI API key.** An early guard cell will stop and print instructions if the key is missing.

## What you'll learn

- Why deterministic graders mis-score open-ended outputs, motivating the need for a judge.
- How to set `OPENAI_API_KEY` and what the guard cell does when it is absent.
- How to define `make_llm_judge(rubric, client)` — a factory that returns a grader with the same `(example, output) -> Score` signature as every other grader in this series.
- How to write a concrete rubric, run the judge on good and bad outputs, and read the `Score` with its `rationale` field.
- How to plug `make_llm_judge` into `run_eval` alongside deterministic graders.
- How to define `judge_pairwise(client, prompt, output_a, output_b)` and why relative judgments are often more reliable than absolute scores.
- The three main traps: **position bias**, **verbosity bias**, and **self-preference / non-determinism** — with a concrete demonstration of position bias.
- How to validate the judge against a small human-labeled set and compute an agreement rate, so the judge is not just another untested component.

## 1. Setup + API Key Guard

This notebook uses `openai` for the LLM judge. Install it if needed:

```bash
pip install openai
```

The cells below (a) import everything and re-declare the shared harness primitives inline, then (b) check for `OPENAI_API_KEY`. **If the key is missing, the guard cell prints setup instructions and raises `SystemExit` — all subsequent API-calling cells are safe to skip.**

To get an API key: visit https://platform.openai.com/api-keys, create a key, and export it in your shell before launching Jupyter:

```bash
export OPENAI_API_KEY="sk-..."
```

Anthropic users: you can swap in `anthropic` SDK calls with the same pattern — the `make_llm_judge` factory accepts any callable you pass as `client`. The default model shown here is `"gpt-4o-mini"` (inexpensive and good enough for grading).

In [ ]:
# ── stdlib ────────────────────────────────────────────────────────────────────
import os
import json
from dataclasses import dataclass, field
from typing import Any, Callable

# ── Load OPENAI_API_KEY from a .env file if present; real env vars still win.
from dotenv import load_dotenv
load_dotenv()

# ── openai SDK ────────────────────────────────────────────────────────────────
from openai import OpenAI  # pip install openai

# ══════════════════════════════════════════════════════════════════════════════
# Shared harness — re-declared inline so this notebook is self-contained.
# These match the canonical definitions from notebooks 03 & 04 exactly.
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Score:
    key: str
    score: float          # normalised to [0, 1]
    passed: bool
    comment: str = ""


@dataclass
class Example:
    input: Any
    expected: Any = None
    metadata: dict = field(default_factory=dict)


@dataclass
class ExampleResult:
    example: Example
    output: Any
    scores: list[Score] = field(default_factory=list)


@dataclass
class EvalReport:
    results: list[ExampleResult]

    @property
    def pass_rate(self) -> float:
        all_scores = [s for r in self.results for s in r.scores]
        if not all_scores:
            return 0.0
        return sum(s.passed for s in all_scores) / len(all_scores)

    @property
    def mean_score(self) -> float:
        all_scores = [s for r in self.results for s in r.scores]
        if not all_scores:
            return 0.0
        return sum(s.score for s in all_scores) / len(all_scores)

    def summary_table(self) -> None:
        """Print a simple per-example summary table."""
        print(f"{'#':<4} {'passed':<8} {'mean score':<12} {'comment'}")
        print("-" * 60)
        for i, r in enumerate(self.results):
            scores = r.scores
            if not scores:
                print(f"{i:<4} {'—':<8} {'—':<12} no graders")
                continue
            passed = all(s.passed for s in scores)
            mean = sum(s.score for s in scores) / len(scores)
            comments = "; ".join(s.comment for s in scores if s.comment)
            print(f"{i:<4} {'✓' if passed else '✗':<8} {mean:<12.3f} {comments[:60]}")
        print("-" * 60)
        print(f"Pass rate: {self.pass_rate:.1%}   Mean score: {self.mean_score:.3f}")


Grader = Callable[[Example, Any], Score]


def run_eval(
    agent: Callable[[Any], Any],
    dataset: list[Example],
    graders: list[Grader],
) -> EvalReport:
    """Run `agent` on every `Example`, apply every grader, return a report."""
    results: list[ExampleResult] = []
    for example in dataset:
        output = agent(example.input)
        scores = [grader(example, output) for grader in graders]
        results.append(ExampleResult(example=example, output=output, scores=scores))
    return EvalReport(results=results)


print("Harness re-declared OK.")

In [ ]:
# ── API key guard ─────────────────────────────────────────────────────────────
# This cell must run before any cell that calls the OpenAI API.
# If OPENAI_API_KEY is not set, it prints setup instructions and stops the
# notebook so subsequent cells that call the API are safe to skip.

_api_key = os.getenv("OPENAI_API_KEY")

if not _api_key:
    print(
        "┌─────────────────────────────────────────────────────────────────┐\n"
        "│  OPENAI_API_KEY is not set.                                     │\n"
        "│                                                                 │\n"
        "│  This is the first notebook in the series that needs a key.    │\n"
        "│  Steps:                                                         │\n"
        "│    1. Visit https://platform.openai.com/api-keys               │\n"
        "│    2. Create a new secret key.                                  │\n"
        "│    3. In your terminal (before launching Jupyter):              │\n"
        "│         export OPENAI_API_KEY=\"sk-...\"                         │\n"
        "│    4. Restart the Jupyter kernel and re-run from the top.       │\n"
        "│                                                                 │\n"
        "│  Anthropic alternative: replace `from openai import OpenAI`    │\n"
        "│  with `import anthropic` and adapt the client calls in         │\n"
        "│  make_llm_judge / judge_pairwise to use the Messages API.       │\n"
        "└─────────────────────────────────────────────────────────────────┘"
    )
    raise SystemExit(
        "Set OPENAI_API_KEY and restart the kernel to continue."
    )

client = OpenAI()  # reads OPENAI_API_KEY from environment automatically
DEFAULT_MODEL = "gpt-4o-mini"

print(f"OpenAI client ready. Default model: {DEFAULT_MODEL}")
print("Key found (first 8 chars):", _api_key[:8] + "…")